# FSCI 396: Survey Validation

------------------
Welcome to today's tutorial! 
Today we'll use Python in Jupyter Notebook to perform some common statistical analyses.
We will cover: Validating surveys.

The goal is to see how we can analyze data with Python, just like we discussed in lectures.


### Steps to Debugging 

I am here to help you! But before you contact me, make sure you have tried the following:
1. Ensure that all brackets and parentheses are paired.
2. Ensure that your code does not have any typos (eg. when calling your data file).
3. Ensure you did not add additional spaces (eg. when calling your data file).
4. You have restarted your kernel and reun your code. 
5. You have tried to understand the error message. 

If you have done all of these things, I am very happy to help! 

In [ ]:
# Import Modules
# ---------------

# Modules in Python are like library books. 
# Each book contains a set of instructions or “recipes” for doing specific tasks.
# - pandas: for handling and analyzing tables of data
# - numpy: for numerical calculations

import pandas as pd
import numpy as np
from scipy import stats
from factor_analyzer import FactorAnalyzer


# Ignore Warnings for simplicity
import warnings
warnings.filterwarnings('ignore')

# Quick check to see if modules loaded correctly
print("Modules loaded successfully!")


In [ ]:
# Define some functions 

def cronbach_alpha(df_items):
    item_vars = df_items.var(axis=0, ddof=1)
    total_var = df_items.sum(axis=1).var(ddof=1)
    n_items = df_items.shape[1]
    return (n_items / (n_items - 1)) * (1 - item_vars.sum() / total_var)

def discriminant_validity(df, scale_columns):
    # Select the scale columns and drop missing rows
    scales = df[scale_columns].dropna()

    # Compute correlation matrix
    corr_matrix = scales.corr()

    # Compute average correlation for each scale with the other scales
    avg_corr = pd.DataFrame({
        'AverageCorrelation': corr_matrix.apply(lambda x: (x.sum() - 1)/(len(x)-1))
    })

    # Combine correlation matrix with average correlation column
    discriminant_table = pd.concat([corr_matrix, avg_corr], axis=1)

    return discriminant_table.round(2)



In [ ]:
# Load datafile/dataframe (df) from CSV
# --------------------------------------

# Here we load our data. Make sure the CSV file is in the same folder as this notebook.

# If your data is .csv, use these two lines: 
filename = 'SampleData.csv'    # Name of your file
df = pd.read_csv(filename)         # Read the CSV into a DataFrame

# Checkpoint to ensure the file has been loaded 
print(filename, 'has been loaded')


# Survey Validation

- **Convergent Validity** : Examines whether theoretically related items correlate as expected, supporting that the scale measures a coherent construct rather than unrelated features. \
       *Measure with Average Variance Extracted (AVE) : proportion of variance explained by the underlying construct.* \
       *AVE > 0.50 --> good ; construct explains most item variance).* \
       *AVE < 0.50 --> Less than half of the variance in the items is explained by the underlying construct. Items may not adequately represent the construct.*

- **Internal Consistency** : Answers the question, “Do these items measure the same thing?” \
       *Measured with Cronbach’s alpha --> checks whether items designed to measure a construct are consistent with each other.* \
       *alpha > 0.70 --> acceptable reliability.* \
       *alpha > 0.80 --> good reliability.* \
       *alpha > 0.95 --> possible item redundancy.*

- **Construct Validity** : Checks whether the survey items group together as expected. \
       *Measured with Exploratory Factor Analysis (EFA): identifies underlying factors in teh data.* \
       *Python looks at how students responded to questions, and sorts the questions into the number of expected constructs the number indicates how well it fits under the factor.*

- **Discriminant Validity** : Checks whether constructs that should be different are actually distinct. \
       *Interscale correlations --> degree of overlap between constructs.* \
       *Low-to-moderate correlations --> good discriminant validity.* \
       *Very high correlations --> constructs may not be meaningfully distinct.*


In [ ]:
# Convergent Validity

# Example: Testing CW# Questions converge to one construct (ie. the three questions converge on Cognitive worry)
measurement = df[['CW1', 'CW2', 'CW3']].dropna()

fa = FactorAnalyzer(n_factors=1, rotation=None)
fa.fit(measurement)

loadings = fa.loadings_
loadings_df = pd.DataFrame( loadings,
                            index=measurement.columns,
                            columns=['Factor Loading'] )

ave = np.mean(loadings.flatten()**2)

display(loadings_df.round(3))
print(f"The average value = {ave:.3f}")


In [ ]:
# Internal Consistency: Cronbach Alpha 

# Exmaple: CW1, CW2, CW3 are all supposed to measure Cognitive Worry.  
# A student who scores high on CW1 should also tend to score high on CW2 and CW3.

measurement = df[['CW1', 'CW2', 'CW3']].dropna()
alpha = cronbach_alpha(measurement)

print(f"Cronbach's alpha = {alpha:.3f}")

In [ ]:
# Construct Validity: is our grouping of constructs actually valid?

# For example, you can see CW# have relatively high values under Factor 4, indicating they belong to the same factor. 

items = df[['CW1', 'CW2', 'CW3',
            'EM1', 'EM2', 'EM3', 'EM4',
            'DS1', 'DS2', 'DS3', 'DS4', 'DS5']].dropna()

# Look for x constructions 
n = 3

FactorAnalysis = FactorAnalyzer(n_factors=n, rotation='varimax')
FactorAnalysis.fit(items)

loadings = pd.DataFrame(
    FactorAnalysis.loadings_,
    index=items.columns,
    columns=['Factor1', 'Factor2', 'Factor3'])

display(loadings.round(2))

In [ ]:
# Discriminant Validty (Correlations)

# Look at the last column for the average correlation 
# A high value indicates the measure/field overlaps with others and may not be measuring a unique construct.

Fields = ['CognitiveWorry', 'Emotionality', 'Distraction', 'SelfEfficacy', 'State_Anxiety']

DV= discriminant_validity(df, Fields)
DV
